# Практическая работа по Anomaly Detection


Цель работы - Реализовать алгоритм обнаружения аномалий на основе многомерного гауссовского распределения для выявления неисправных серверов в сети.

In [1]:
import numpy as np

## 1. Теоретическая основа

### 1.1 Что такое обнаружение аномалий?
Обнаружение аномалий - это задача идентификации редких событий или наблюдений, которые значительно отличаются от большинства данных.

#### Применения:

1. Обнаружение мошенничества в финансовых транзакциях
2. Выявление неисправностей в производственном оборудовании
3. Мониторинг серверов и сетевого оборудования
4. Обнаружение вторжений в компьютерных системах

### 1.2 Гауссовское (нормальное) распределение
Плотность вероятности для одномерного нормального распределения: $$p(x; \mu, \sigma^2) = \frac{1} {\sqrt{2\pi\sigma^2}} \exp(-\frac{(x - \mu)^2} {2\sigma^2})$$

- $\mu$ - Математическое ожидание (среднее значение)
- $\sigma^2$ - Дисперсия
- $\sigma$ - Среднеквадратичное отклонение

### 1.3 Многомерное гауссовское распределение
$$f(\mathbf{x}) = \frac{1}{\sqrt{(2\pi)^k |\boldsymbol{\Sigma}|}} \exp\left(-\frac{1}{2}(\mathbf{x} - \boldsymbol{\mu})^T \boldsymbol{\Sigma}^{-1} (\mathbf{x} - \boldsymbol{\mu})\right)
$$

- $x$ - вектор случайных величин размерности $k$.
- $\mu$ - вектор математических ожиданий (средних значений).
- $\sum$ - ковариационная матрица размером $k * k$.
- $|\sum|$ - определитель (детерминант) ковариационной матрицы.
- $\sum^{-1}$ - обратная ковариационная матрица.
- $k$ - размерность пространства.

> Если все компоненты вектора $x$ независимы, то матрица будет диагональной, а формула упростится до произведения плотностей одномерных распределений.

В нашем случае используем диагональную ковариационную матрицу.

## 2. Реализация функций

### 2.1 Функция `estimate_gaussian` - оценка параметров распределения


In [2]:
def estimate_gaussian(X):
    """
    Оценивает параметры гауссовского распределения для каждого признака

    Параметры:
        X: (m, n) - матрица данных (m примеров, n признаков)

    Возвращает:
        mu: (n,) - среднее значение для каждого признака
        var: (n,) - дисперсия для каждого признака
    """
    m, n = X.shape

    # Векторизованное вычисление среднего
    # mu_i = (1/m) * sum_{j=1 to m} x_i^{(j)}
    mu = (1 / m) * np.sum(X, axis=0)

    # Векторизованное вычисление дисперсии
    # var_i = (1/m) * sum_{j=1 to m} (x_i^{(j)} - mu_i)^2
    var = (1 / m) * np.sum((X - mu) ** 2, axis=0)

    return mu, var

In [3]:
# Например, для 3 примеров с 2 признаками
X = np.array([[1, 4],
              [2, 5],
              [3, 6]])

mu, var = estimate_gaussian(X)
print(mu, var)
# mu = [2, 5]
# var = [2/3, 2/3] = [0.667, 0.667]

[2. 5.] [0.66666667 0.66666667]


### 2.2 Функция `select_threshold` - выбор порога
Эта функция выбирает оптимальный порог $\epsilon$ на основе F1-score для классификации аномалий. Алгоритм старается максимизировать F1.

#### Пояснения метрик:
1. True Positive (TP): алгоритм сказал "аномалия" и это действительно аномалия
2. False Positive (FP): алгоритм сказал "аномалия", но это нормальный объект
3. False Negative (FN): алгоритм сказал "норма", но это аномалия

Precision (Точность):
$$Precision = \frac{TP} {TP + FP}$$
> Из всех объектов, которые алгоритм пометил как аномалии, какая доля действительно является аномалиями?

Recall (Полнота):
$$Recall = \frac{TP} {TP + FN}$$
> Из всех реальных аномалий, какую долю алгоритм смог найти?

F1-мера:
$$F1 = 2 * \frac{Precision * Recall} {Precision + Recall} $$

In [4]:
def select_threshold(y_val, p_val):
    """
    Выбирает оптимальный порог на основе F1-меры

    Параметры:
        y_val: (m,) - истинные метки (1 - аномалия, 0 - норма)
        p_val: (m,) - вероятности для валидационной выборки

    Возвращает:
        epsilon: float - оптимальный порог
        best_F1: float - лучшее значение F1-меры
    """
    best_epsilon = 0
    best_F1 = 0
    step_size = (max(p_val) - min(p_val)) / 1000

    for epsilon in np.arange(min(p_val), max(p_val), step_size):
        # Предсказания: аномалия, если вероятность < порога
        predictions = (p_val < epsilon)

        # Вычисляем метрики
        tp = np.sum((predictions == 1) & (y_val == 1))  # True Positives
        fp = np.sum((predictions == 1) & (y_val == 0))  # False Positives
        fn = np.sum((predictions == 0) & (y_val == 1))  # False Negatives

        # Точность (Precision) - доля правильных аномалий среди всех предсказанных
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0

        # Полнота (Recall) - доля найденных аномалий среди всех реальных
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0

        # F1-мера - гармоническое среднее точности и полноты
        F1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0

        if F1 > best_F1:
            best_F1 = F1
            best_epsilon = epsilon

    return best_epsilon, best_F1